# 07 · Explanations

Day 7 check: do the drivers, the sentences and the comparable films make sense, and are the cut-offs right?

1. Strength cut-offs: how often a driver reads "strong" (2016–18, ≤2015 models)
2. Which groups drive each target
3. Inside the studio group for budget
4. Comparable-film weights, measured on 2016–18
5. Famous films through `predict()`

Sections 1–4 use the ≤2015 models from `make train` on the 2016–18 validation films, so nothing is
fitted on what is measured. Section 5 uses the app's all-data models, which were trained on these films,
so their ranges look better than they would for a new film; it is there to read the explanations,
not to judge accuracy. The test years (2019+) are only shown, never used to choose anything.

In [1]:
import math
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

from ml.data.build_db import DB_FILE
from ml.eval.splits import load
from ml.explain import comparables as C
from ml.explain.drivers import CUTOFFS, contributions, group_contributions
from ml.explain.groups import GROUPS, feature_groups
from ml.features.build import FeatureContext, FilmForms, film_to_form
from ml.features.contract import GENRES, validate
from ml.models.chain import format_money
from ml.models.train import load_model
from ml.predict import InvalidForm, Predictor

G = feature_groups()
OLD = Path("ml/artifacts")  # the <=2015 models from make train
grouped = {}
for target in ("revenue", "budget", "score"):
    model, val = load_model(target, OLD), load(target, "validation")
    grouped[target] = group_contributions(contributions(model, val), G).drop(columns="expected")
{t: len(g) for t, g in grouped.items()}

{'revenue': 886, 'budget': 1162, 'score': 3600}

## 1. Strength cut-offs

A money contribution c means "times e^c". The plan started at strong ≥ +25%, moderate ≥ +10%. The
table shows what each choice does to the top 5 drivers of every 2016–18 film.

In [2]:
def top5(g):
    return np.sort(np.abs(g.to_numpy()), axis=1)[:, ::-1][:, :5]

sizes = pd.DataFrame({t: np.median(top5(g), axis=0) for t, g in grouped.items()},
                     index=[f"driver {i}" for i in range(1, 6)])
sizes.assign(**{f"{t} as %": [f"{math.expm1(x):+.0%}" for x in sizes[t]] for t in ("revenue", "budget")}).round(3)

,revenue,budget,score,revenue as %,budget as %
driver 1,0.713,0.665,0.318,+104%,+94%
driver 2,0.395,0.437,0.180,+48%,+55%
driver 3,0.300,0.262,0.116,+35%,+30%
driver 4,0.235,0.194,0.082,+26%,+21%
driver 5,0.183,0.153,0.061,+20%,+17%


In [3]:
rows = []
for target in ("revenue", "budget"):
    top = top5(grouped[target])
    for name, (strong, moderate) in {"+25% / +10% (plan)": (1.25, 1.10), "+50% / +20% (chosen)": (1.5, 1.2),
                                     "2x / +25%": (2.0, 1.25)}.items():
        s, m = top >= math.log(strong), (top >= math.log(moderate)) & (top < math.log(strong))
        rows.append({"target": target, "cut-offs": name, "strong": s.mean(), "moderate": m.mean(),
                     "slight": 1 - s.mean() - m.mean(), "top driver strong": s[:, 0].mean()})
top = top5(grouped["score"])
s, m = top >= 0.3, (top >= 0.1) & (top < 0.3)
rows.append({"target": "score", "cut-offs": "0.3 / 0.1 points", "strong": s.mean(), "moderate": m.mean(),
             "slight": 1 - s.mean() - m.mean(), "top driver strong": s[:, 0].mean()})
pd.DataFrame(rows).set_index(["target", "cut-offs"]).style.format("{:.0%}")

With the plan's +25% / +10%, 99% of top revenue drivers and 72% of all listed drivers read "strongly",
so the word carries no information. **Chosen (Dimitris): +50% / +20%** for revenue and budget; the
score keeps 0.3 / 0.1 points. `CUTOFFS` in `ml/explain/drivers.py` now holds:

In [4]:
{t: [f"{math.expm1(x):+.0%}" if t != "score" else x for x in v] for t, v in CUTOFFS.items()}

{'revenue': ['+50%', '+20%'], 'budget': ['+50%', '+20%'], 'score': [0.3, 0.1]}

## 2. Which groups drive each target

In [5]:
share = {}
for target, g in grouped.items():
    top = g.abs().to_numpy().argsort(axis=1)[:, ::-1][:, :5]
    first = pd.Series(g.columns[top[:, 0]]).value_counts(normalize=True)
    any5 = pd.Series(g.columns[top.ravel()]).value_counts() / len(g)
    share[(target, "top driver")] = first
    share[(target, "in the top 5")] = any5
table = pd.DataFrame(share).fillna(0).rename(index=GROUPS)
table.sort_values(("revenue", "in the top 5"), ascending=False).style.format("{:.0%}")

Budget drives revenue (as on Day 5), the studio and cast drive budget, and genre, runtime and cast
drive score. The studio often leads budget, which section 3 explains.

## 3. Inside the studio group for budget

In [6]:
model, val = load_model("budget", OLD), load("budget", "validation")
c = contributions(model, val)
studio = [f for f in model.features if G[f] == "studio"]
(c[studio].abs().sum() / c[studio].abs().sum().sum()).sort_values(ascending=False).head(5).round(3)

company_budget                  0.896
major_studio                    0.064
company_new_line_cinema         0.010
company_tf1_films_production    0.006
company_unknown                 0.005
dtype: float64

Almost all of it (about 90%) is `company_budget`, the biggest earlier budget among the film's companies
(Day 6), not the company yes/no columns. "Marvel Studios as the studio strongly raises the expected
budget" is therefore about Marvel's earlier budgets, which is how the sentence reads anyway.

## 4. Comparable-film weights

600 films from 2016–18 with a real revenue, each judged at its own release date with its own genres,
themes, language, synopsis and budget. For each weight set: how far the comparables' median revenue
and mean score are from the film's own, and how many comparables share its language or are well
known (200+ votes). Budget closeness is part of the similarity, so a budget comparison would be
circular and isn't shown.

In [7]:
con = sqlite3.connect(DB_FILE)
index = C.Index.load()
votes = pd.read_sql("SELECT vote_count FROM movies WHERE score_ok ORDER BY id", con)["vote_count"].to_numpy(float)
fame = np.log1p(votes) / np.log1p(votes.max())
emb = np.load("data/overview_embeddings.npz")
vec_row = {int(i): k for k, i in enumerate(emb["ids"])}
pos = {int(i): k for k, i in enumerate(index.ids)}
val_films = pd.read_sql("""SELECT id, year, original_language, vote_average,
        CASE WHEN money_suspect THEN NULL ELSE budget_2025 END AS budget, revenue_2025 AS revenue
    FROM movies WHERE score_ok AND year BETWEEN 2016 AND 2018 AND revenue_2025 IS NOT NULL AND NOT money_suspect
    ORDER BY id""", con)
val_films = val_films[val_films["id"].isin(pos)].sample(600, random_state=0)


def measure(weights, w_fame=0.0):
    old = dict(C.WEIGHTS)
    C.WEIGHTS.clear()
    C.WEIGHTS.update(weights)
    rev, score, lang, famous = [], [], [], []
    try:
        for r in val_films.itertuples():
            k = pos[r.id]
            genres = [g for g, on in zip(GENRES, index.genres[k]) if on]
            themes = [t for t, on in zip(index.theme_names, index.themes[k]) if on]
            vector = emb["vectors"][vec_row[r.id]] if r.id in vec_row else None
            log_b = math.log(r.budget) if pd.notna(r.budget) else None
            language = r.original_language if "language" in weights else None
            sim = C.similarity(index, genres, themes, language, vector, int(r.year), log_b)
            if w_fame:
                sim = (sim + w_fame * fame) / (1 + w_fame)
            sim[index.days >= index.days[k]] = -np.inf
            top = np.argpartition(-sim, C.TOP_POOL - 1)[:C.TOP_POOL]
            top = top[np.argsort(-sim[top], kind="stable")]
            money = ~np.isnan(index.budget[top]) | ~np.isnan(index.revenue[top])
            chosen = np.array([*top[money][:5], *top[~money][:max(0, 5 - int(money.sum()))]])
            known = index.revenue[chosen][~np.isnan(index.revenue[chosen])]
            if len(known):
                rev.append(abs(np.median(np.log(known)) - math.log(r.revenue)))
            score.append(abs(index.score[chosen].mean() - r.vote_average))
            lang.append((index.languages[chosen] == r.original_language).mean())
            famous.append((votes[chosen] >= 200).mean())
    finally:
        C.WEIGHTS.clear()
        C.WEIGHTS.update(old)
    return {"revenue MAE log": np.mean(rev), "score MAE": np.mean(score), "same language": np.mean(lang),
            "200+ votes": np.mean(famous)}


PLAN = {"genres": 0.3, "themes": 0.2, "synopsis": 0.3, "era": 0.1, "budget": 0.1}
variants = {
    "plan (no language, budget 0.1)": (PLAN, 0),
    "+ language 0.1": ({**PLAN, "language": 0.1}, 0),
    "budget 0.2": ({**PLAN, "budget": 0.2}, 0),
    "+ language 0.1, budget 0.2 (chosen)": (dict(C.WEIGHTS), 0),
    "chosen + well-known tie-break 0.1": (dict(C.WEIGHTS), 0.1),
    "synopsis 0.15": ({**PLAN, "synopsis": 0.15}, 0),
}
results = pd.DataFrame({name: measure(*args) for name, args in variants.items()}).T
results.style.format({"revenue MAE log": "{:.3f}", "score MAE": "{:.3f}", "same language": "{:.0%}",
                      "200+ votes": "{:.0%}"})

,revenue MAE log,score MAE,same language,200+ votes
"plan (no language, budget 0.1)",1.883,0.602,60%,73%
+ language 0.1,1.875,0.586,85%,74%
budget 0.2,1.833,0.599,60%,73%
"+ language 0.1, budget 0.2 (chosen)",1.802,0.584,84%,74%
chosen + well-known tie-break 0.1,1.931,0.572,83%,83%
synopsis 0.15,1.903,0.622,57%,73%


**Chosen (Dimitris): add language (0.1) and double budget (0.2).** The comparables' revenue and score
come closer to the film's own, and most comparables now share its language. The well-known tie-break
gives more famous comparables but pulls their revenue further away (famous films earn more), so it
was dropped. Halving the synopsis weight makes every measure worse, so it stays at 0.3.

## 5. Famous films through `predict()`

Each film is written as a creator's form (`film_to_form`) and judged at the 1st of its release month,
as the app judges a creator's film. The all-data models were trained on these films, so treat the
ranges as illustrations. *Parasite* (2019) is a test-year film: shown only.

In [8]:
predictor = Predictor.load()
ctx, films = predictor.ctx, FilmForms.from_db(con)
TITLES = [("Get Out", 2017), ("La La Land", 2016), ("Avengers: Infinity War", 2018),
          ("The Shape of Water", 2017), ("Paddington 2", 2017), ("Parasite", 2019)]


def show(title, year):
    movie_id, budget, revenue, score = con.execute(
        "SELECT id, budget_2025, revenue_2025, vote_average FROM movies WHERE title = ? AND year = ?",
        (title, year)).fetchone()
    form = film_to_form(movie_id, films, ctx)
    form["actors"] = form["actors"][:10]
    try:
        out = predictor.predict(form)
    except InvalidForm as e:
        print(f"{title}: not a valid creator form ({e.errors})")
        return
    print(f"=== {title} ({year}): real budget {format_money(budget) if budget else '-'}, "
          f"revenue {format_money(revenue) if revenue else '-'}, score {score}")
    for t in ("budget", "revenue", "score"):
        r = out[t]
        extra = f", {r['mode']}" if t == "revenue" else ""
        print(f"  {t}: {r['text']} (middle {r['middle_text']}{extra}{', rough' if r['rough'] else ''})")
    print("  " + out["budget"]["comparison"].get("sentence", out["budget"]["comparison"]["text"]))
    for t in ("revenue", "score"):
        d = out["drivers"][t]
        for s in d["up"] + d["down"]:
            print(f"  [{t} {s['direction']} {s['strength']}] {s['text']}")
    print("  comparables: " + "; ".join(f"{f['title']} ({f['year']})" for f in out["comparables"]))
    for w in out["warnings"]:
        print(f"  warning: {w['text']}")
    print()


for title, year in TITLES:
    show(title, year)

=== Get Out (2017): real budget $5.9M, revenue $340M, score 7.625
  budget: $1.1M–$18M (middle $5.1M, rough)
  revenue: $160K–$59M (middle $3.5M, given, rough)
  score: 5.3–7.3 (middle 6.4)
  Your budget of $5.9M is within what similar films usually cost
  [revenue up slight] The genres Mystery, Thriller and Horror slightly raise the expected revenue.
  [revenue down strong] A budget of $5.9M strongly lowers the expected revenue.
  [revenue down strong] The cast's earlier films (led by Daniel Kaluuya) strongly lower the expected revenue.
  [revenue down strong] Blumhouse Productions, QC Entertainment and Monkeypaw Productions as studios strongly lower the expected revenue.
  [revenue down moderate] Jordan Peele, with no earlier films as director in our data, lowers the expected revenue.
  [score up moderate] The themes "murder and revenge", "comedy and satire" and "politics and religion" raise the expected audience score.
  [score up moderate] A runtime of 104 minutes raises the expect

=== Avengers: Infinity War (2018): real budget $380M, revenue $2.6B, score 8.245
  budget: $220M–$570M (middle $390M)
  revenue: $1.1B–$4.6B (middle $2.1B, given)
  score: 6.4–8.3 (middle 7.5)
  Your budget of $380M is within what similar films usually cost
  [revenue up strong] A budget of $380M strongly raises the expected revenue.
  [revenue up strong] Being part of The Avengers Collection (2 earlier films) strongly raises the expected revenue.
  [revenue up strong] Marvel Studios as the studio strongly raises the expected revenue.
  [revenue up moderate] A runtime of 149 minutes raises the expected revenue.
  [revenue up moderate] A PG-13 rating raises the expected revenue.
  [score up strong] A runtime of 149 minutes strongly raises the expected audience score.
  [score up moderate] The cast's earlier films (led by Robert Downey Jr.) raise the expected audience score.
  [score up moderate] The themes "based on a book", "science fiction" and 2 more raise the expected audience score

=== Paddington 2 (2017): real budget $53M, revenue $380M, score 7.488
  budget: $36M–$150M (middle $76M)
  revenue: $160M–$780M (middle $310M, given)
  score: 5.9–7.5 (middle 6.7)
  Your budget of $53M is within what similar films usually cost
  [revenue up strong] A budget of $53M strongly raises the expected revenue.
  [revenue up strong] Being part of Paddington Collection (1 earlier film) strongly raises the expected revenue.
  [revenue up strong] Heyday Films, StudioCanal and Anton Capital Entertainment as studios strongly raise the expected revenue.
  [revenue up moderate] The genres Adventure, Comedy and Family raise the expected revenue.
  [revenue up moderate] Paul King's 2 earlier films raise the expected revenue.
  [score up moderate] The cast's earlier films (led by Ben Whishaw) raise the expected audience score.
  [score up moderate] Releasing in 2017 raises the expected audience score.
  [score up slight] A PG rating slightly raises the expected audience score.
  [score u

## Conclusions

- **Drivers read sensibly:** *Avengers: Infinity War* is driven up by its budget, the franchise and
  Marvel; *Get Out* down by its small budget, a first-time director and small studios; *La La Land*'s
  score up by Damien Chazelle's earlier films.
- **Cut-offs changed** from the plan's +25% / +10% to +50% / +20% (money), so "strongly" is kept for
  the large effects. Score stays at 0.3 / 0.1 points.
- **Comparables changed** from the plan's weights: language added (0.1) and budget doubled (0.2), based
  on 600 films from 2016–18. No well-known tie-break.
- **The studio driver for budget is the company's earlier budgets** (about 90% of the group).
- **Breakouts stay outside the range:** *Get Out* earned $340M against $160K–$59M, even though the
  all-data models saw it in training. The drivers explain the typical film like it (small budget, new
  director), not the breakout.
- **Some comparables are still obscure:** *Get Out*'s are small horror films scored around 4.5–6. The
  well-known tie-break would fix that but makes the comparables' revenue less like the film's own.
- *Parasite* (test year, shown only) now gets a Korean comparable (*The Yellow Sea*) through the language weight.
